# Portfoy + Benchmark Dashboard
Kisisel portfoyunuzu benchmark'larla karsilastirin.
Asagidaki formdan islem ekleyebilirsiniz — otomatik CSV'ye kaydedilir ve dashboard guncellenir.

In [ ]:
# Hucre 1 - Bagimlilik kurulumu
import subprocess, sys

REQUIRED = ["yfinance", "plotly", "ipywidgets"]
for pkg in REQUIRED:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Bagimliliklar hazir")

In [ ]:
# Hucre 2 - Importlar + Google Drive baglama
import os, sys
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive, output
    drive.mount('/content/drive')
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
    print("Google Drive baglandi | Colab widget/Plotly renderer hazir")
else:
    print("Yerel ortam - Drive mount atlandi")

In [ ]:
# Hucre 3 - Konfigurasyon
PROJECT_ROOT_CANDIDATES = [
    os.getcwd(),
    "/content/BenchmarkTakip",
    "/content/drive/MyDrive/PortfolioProject",
]
PROJECT_ROOT = next(
    (p for p in PROJECT_ROOT_CANDIDATES if os.path.isdir(os.path.join(p, "lib"))),
    os.getcwd(),
)

DRIVE_DATA_DIR = "/content/drive/MyDrive/PortfolioProject"
DRIVE_BASE = os.environ.get("PORTFOLIO_DATA_DIR")
if not DRIVE_BASE:
    if IN_COLAB and os.path.isdir(DRIVE_DATA_DIR):
        DRIVE_BASE = DRIVE_DATA_DIR
    else:
        DRIVE_BASE = os.path.join(PROJECT_ROOT, "data")
DRIVE_BASE = os.path.abspath(DRIVE_BASE) + os.sep

CACHE_PATH         = os.path.join(DRIVE_BASE, "cache")
TRANSACTIONS_PATH  = os.path.join(DRIVE_BASE, "transactions.csv")

SYMBOLS = {
    "Gram Altin": "GC=F",
    "Gram Gumus": "SI=F",
    "DOLAR":      "USDTRY=X",
    "EURO":       "EURTRY=X",
    "BIST100":    "XU100.IS",
}

TCMB_POLICY_RATE_PCT = 37
DASHBOARD_MIN_DATE = "2015-01-01"  # Date picker / benchmark 10-yil alt sinir
DEFAULT_END = datetime.today().strftime("%Y-%m-%d")

print(f"Config hazir | PROJECT_ROOT={PROJECT_ROOT} | Islem dosyasi: {TRANSACTIONS_PATH}")

In [ ]:
# Hucre 4 - lib/ import
import importlib, sys

LIB_PATH = os.path.join(PROJECT_ROOT, "lib")
if not os.path.isdir(LIB_PATH):
    raise FileNotFoundError(
        f"lib klasoru bulunamadi: {LIB_PATH}. Colab'da once repoyu clone edip os.chdir(repo_klasoru) yapin."
    )

if LIB_PATH not in sys.path:
    sys.path.insert(0, LIB_PATH)

# lib/ modullerini reload et (dosya degisikliklerini kernel restart olmadan yakalar)
_lib_modules = [
    "data_loader", "portfolio_engine", "benchmark_engine",
    "chart_builder", "portfoy_dashboard",
]
for _m in _lib_modules:
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

# Paylasilan moduller (BenchmarkKarsilastirma.ipynb ile ortak)
from data_loader import (
    fetch_prices, load_transactions_csv, load_cpi_series, load_tcmb_rates
)
from portfolio_engine import (
    compute_wac, compute_portfolio_value_series,
    compute_asset_contributions
)
from benchmark_engine import build_benchmark_series, build_deposit_series
from chart_builder import (
    build_performance_line_chart, build_donut_chart,
    build_kpi_cards, build_summary_table
)

# PortfolyoBenchmark.ipynb'ye ozel (BenchmarkKarsilastirma'dan bagımsız)
from portfoy_dashboard import (
    create_date_range_picker, create_currency_toggle, wire_dashboard,
    create_transaction_form_v3, create_portfolio_viewer,
    extract_stock_symbols,
    compute_portfolio_performance_index,
)

print("lib/ moduller yuklendi")

In [ ]:
# Hucre 5 - Veri yukle
os.makedirs(CACHE_PATH, exist_ok=True)

CPI_PATH  = os.path.join(DRIVE_BASE, "cpi_turkey.csv")
TCMB_PATH = os.path.join(DRIVE_BASE, "tcmb_rates.csv")

cpi_series  = load_cpi_series(CPI_PATH)
tcmb_rates  = load_tcmb_rates(TCMB_PATH, policy_rate_pct=TCMB_POLICY_RATE_PCT)

# Portfoy ve fiyat verisi yukle
def load_all():
    """transactions.csv'den yeniden yukle ve hesapla.
    Hisse sembolleri transactions'tan otomatik cikarilir (extract_stock_symbols).
    """
    global transactions, prices, fx_usdtry, wac_state, portfolio_values
    global DATA_START, stock_map, FULL_SYMBOL_MAP

    transactions = load_transactions_csv(TRANSACTIONS_PATH)

    if len(transactions) == 0:
        DATA_START = datetime.today().strftime("%Y-%m-%d")
        stock_map = {}
        FULL_SYMBOL_MAP = dict(SYMBOLS)
        prices = pd.DataFrame()
        fx_usdtry = pd.Series(dtype=float)
        wac_state = {}
        portfolio_values = pd.DataFrame()
        print("Henuz islem yok.")
        return DATA_START

    DATA_START  = transactions["Tarih"].min().strftime("%Y-%m-%d")

    # Hisse sembollerini transactions'tan cikar (ASELS -> ASELS.IS vb.)
    stock_map      = extract_stock_symbols(transactions, SYMBOLS.keys())
    FULL_SYMBOL_MAP = {**SYMBOLS, **stock_map}

    all_symbols = list(FULL_SYMBOL_MAP.values()) + ["USDTRY=X"]
    prices      = fetch_prices(all_symbols, start=DASHBOARD_MIN_DATE, end=DEFAULT_END, cache_path=CACHE_PATH)
    fx_usdtry   = prices["USDTRY=X"].dropna()

    wac_state        = compute_wac(transactions)
    portfolio_values = compute_portfolio_value_series(transactions, prices, fx_usdtry, FULL_SYMBOL_MAP)
    return DATA_START

# Globals ilk deger
stock_map = {}
FULL_SYMBOL_MAP = dict(SYMBOLS)

DATA_START = load_all()
print(f"Portfoy yuklendi | Baslangic: {DATA_START} | {len(transactions)} islem | Hisseler: {list(stock_map.keys())}")

In [ ]:
# Hucre 6 - Islem Ekleme Formu (v3)
# ============================================================
# 3 sutun: Varlik alim/satim | Hisse alim/satim | Sermaye yonetimi

dashboard_output = widgets.Output()  # Hucre 7 ile paylasim icin global

def on_transaction_saved():
    """Yeni islem kaydedilince portfoy verisi yeniden yukle ve dashboard'u yenile."""
    load_all()
    with dashboard_output:
        dashboard_output.clear_output(wait=True)
        try:
            render(DATA_START, DEFAULT_END, "TL")
        except NameError:
            print("Dashboard henuz yuklenmedi. Lutfen Hucre 7'yi calistirin.")

# BIST100 haric sabit varliklar sol panele gider; BIST100 benchmark olarak kalir
non_stock_assets = [a for a in SYMBOLS.keys() if a != "BIST100"]

form = create_transaction_form_v3(
    non_stock_assets=non_stock_assets,
    transactions_path=TRANSACTIONS_PATH,
    on_save_callback=on_transaction_saved,
    wac_state_getter=lambda: wac_state,
    reset_callback=load_all,
)

display(form)

In [ ]:
# Hucre 7 - Dashboard + Portfoy Goruntuleyi
CURRENCY_LABELS = {"TL": "TL (Nominal)", "USD": "USD", "REAL": "Reel (TUFE)"}

def render(start_date, end_date, currency):
    if portfolio_values is None or len(portfolio_values) == 0:
        print("Henuz islem yok. Yukardaki formdan islem ekleyin.")
        return

    twrr = compute_portfolio_performance_index(
        portfolio_values.loc[start_date:end_date],
        transactions,
        currency=currency,
        fx_usdtry=fx_usdtry,
        cpi_series=cpi_series,
    )

    # Benchmark sadece sabit SYMBOLS (bireysel hisseler benchmark'a girmez)
    benchmark_df = build_benchmark_series(
        symbols=list(SYMBOLS.values()),
        start_date=start_date,
        end_date=end_date,
        prices=prices,
        fx_usdtry=fx_usdtry,
        cpi_series=cpi_series if currency == "REAL" else None,
        currency=currency,
    )
    # yfinance sembol kodlarini okunakli isimlere cevir (GC=F -> Altin vb.)
    benchmark_df = benchmark_df.rename(columns={v: k for k, v in SYMBOLS.items()})
    deposit_series = build_deposit_series(tcmb_rates, start_date, end_date)
    benchmark_df["Mevduat"] = deposit_series

    currency_label = CURRENCY_LABELS.get(currency, currency)

    contributions = compute_asset_contributions(
        transactions, prices, wac_state, FULL_SYMBOL_MAP,
        start_date=start_date, end_date=end_date,
        fx_usdtry=fx_usdtry,
    )

    period_values = portfolio_values.loc[:end_date]
    last_values   = period_values["asset_values_tl"].iloc[-1] if len(period_values) else portfolio_values["asset_values_tl"].iloc[-1]
    total_val     = sum(last_values.values())
    asset_names   = list(last_values.keys())
    asset_vals    = list(last_values.values())
    asset_weights = [v / total_val for v in asset_vals] if total_val > 0 else [0] * len(asset_vals)

    display(build_performance_line_chart(twrr, benchmark_df, currency_label))
    if total_val > 0:
        display(build_donut_chart(asset_names, asset_weights, asset_vals))
    display(build_kpi_cards(contributions))
    display(build_summary_table(contributions))

    # Portfoy goruntuleyi — FULL_SYMBOL_MAP ile hem sabit varliklar hem hisseler
    display(create_portfolio_viewer(
        transactions=transactions,
        prices=prices,
        wac_state=wac_state,
        symbol_map=FULL_SYMBOL_MAP,
        fx_usdtry=fx_usdtry,
        refresh_callback=load_all,
    ))

start_picker, end_picker = create_date_range_picker(
    min_date=datetime.strptime(DASHBOARD_MIN_DATE, "%Y-%m-%d"),
    max_date=datetime.today(),
    default_start=datetime.strptime(DATA_START, "%Y-%m-%d"),
    default_end=datetime.today(),
)
currency_toggle = create_currency_toggle()

dashboard = wire_dashboard(
    render_fn=render,
    output_widget=dashboard_output,
    start_picker=start_picker,
    end_picker=end_picker,
    currency_toggle=currency_toggle,
)

with dashboard_output:
    render(DATA_START, DEFAULT_END, "TL")

display(dashboard)